# Preprocesamiento CASEN 2024
## Inserción Laboral de Migrantes en Chile
**Fuente:** Encuesta de Caracterización Socioeconómica Nacional (CASEN) 2024 — Ministerio de Desarrollo Social
**Objetivo:** Procesar variables de ingreso, formalidad, pobreza y construir el Índice de Inserción Laboral Multidimensional.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE = os.path.abspath(os.path.join('..', '..'))
RAW_CASEN = os.path.join(BASE, 'input', 'data', 'casen_2024.dta')
OUT_DATA  = os.path.join(BASE, 'output', 'data')
os.makedirs(OUT_DATA, exist_ok=True)
print(f"Cargando CASEN desde: {RAW_CASEN}")

## 1. Carga de datos

In [ ]:
# Cargar sin convertir categóricas para mayor control sobre los códigos
df_raw = pd.read_stata(RAW_CASEN, convert_categoricals=False)
print(f"CASEN 2024: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print()
# Mostrar primeras columnas relevantes
cols_preview = ['id_persona','region','lugar_nac','sexo','edad','activ',
                'yoprcor','o10','contrato','cotiza','pobreza','pobreza_multi']
df_raw[cols_preview].head(4)

## 2. Tratamiento de códigos de valor perdido

In [ ]:
# En CASEN, el código -88 indica 'No aplica' y -99 indica 'No sabe/No responde'
# Reemplazamos por NaN para tratamiento uniforme

MISSING_CODES = [-88, -99, -9, 88, 99]

df = df_raw.copy()
# Solo reemplazar en columnas numéricas
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col] = df[col].replace(MISSING_CODES, np.nan)

print(f"Columnas numéricas procesadas: {len(num_cols)}")
print("Nulos en variables clave:")
clave = ['lugar_nac','activ','yoprcor','o10','contrato','cotiza','pobreza','pobreza_multi']
print(df[clave].isnull().sum().to_string())

## 3. Variable de condición migratoria

In [ ]:
# lugar_nac: 0=Nacido en Chile, 1=Nacido fuera de Chile
df['migrante'] = np.nan
df.loc[df['lugar_nac'] == 0, 'migrante'] = 0
df.loc[df['lugar_nac'] == 1, 'migrante'] = 1
df['migrante'] = df['migrante'].astype('Int64')

print("Condición migratoria:")
print("  Chilenos (lugar_nac=0):", (df['migrante']==0).sum())
print("  Migrantes (lugar_nac=1):", (df['migrante']==1).sum())
print("  Sin información:", df['migrante'].isnull().sum())
print(f"  Tasa migratoria (excl. sin info): {df['migrante'].mean()*100:.1f}%")

In [ ]:
# País de origen de migrantes
paises_top = df[df['migrante']==1]['r1b_pais_esp'].value_counts().head(10)
print("Top 10 países de origen de migrantes CASEN:")
print(paises_top.to_string())

## 4. Variables sociodemográficas

In [ ]:
# Sexo: 1=Hombre, 2=Mujer
df['sexo_str'] = df['sexo'].map({1: 'Hombre', 2: 'Mujer'})

# Grupo de edad
df['grupo_edad'] = pd.cut(
    df['edad'],
    bins=[0, 14, 24, 34, 44, 54, 64, 200],
    labels=['0-14','15-24','25-34','35-44','45-54','55-64','65+']
)

# Macrozona
macrozona_map = {
    1:'Norte_grande', 2:'Norte_grande', 15:'Norte_grande',
    3:'Norte_chico',  4:'Norte_chico',
    5:'Centro',       13:'Centro',
    6:'Sur',          7:'Sur',  16:'Sur',
    8:'Sur',          9:'Sur',  14:'Sur',
    10:'Austral',     11:'Austral', 12:'Austral'
}
df['macrozona'] = df['region'].map(macrozona_map)

print("Grupo de edad:")
print(df['grupo_edad'].value_counts().sort_index().to_string())

## 5. Variables laborales y de ingresos

In [ ]:
# Condición de actividad (activ): 1=Ocupado, 2=Desocupado, 3=Inactivo
df['ocupado']    = (df['activ'] == 1).astype('Int64')
df['desocupado'] = (df['activ'] == 2).astype('Int64')

print("Condición de actividad:")
print(df['activ'].value_counts().to_dict())

In [ ]:
# Nivel educativo: reconstrucción granular desde e6a_no_asiste + e6c_completo
# educc (0-6) no distingue Técnica vs Universitaria dentro de 'Superior'
# Reconstruimos 'educa' con 7 categorías, igual que en CASEN 2022
print("e6a_no_asiste (nivel al que no asiste / ya no cursa):")
print(df['e6a_no_asiste'].value_counts().sort_index().to_dict())
print()
print("e6c_completo (1=Sí completó, 2=No completó):")
print(df['e6c_completo'].value_counts().sort_index().to_dict())

In [ ]:
# Paso 1: categorizar e6a_no_asiste
# 6-7=Básica | 8-11=Media | 12=Técnico | 13=Universitaria | 14-15=Postgrado
df['e6a_cat'] = np.select([
    df['e6a_no_asiste'].between(6, 7),
    df['e6a_no_asiste'].between(8, 11),
    df['e6a_no_asiste'] == 12,
    df['e6a_no_asiste'] == 13,
    df['e6a_no_asiste'].between(14, 15)
], ['Basica', 'Media', 'Tecnico', 'Universitaria', 'Postgrado'], default='Otro')

# Paso 2: educa numérico con distinción completo/incompleto
# e6c_completo: 1=Completo, 2=Incompleto
# 1=Básica, 2=Media, 3=Téc_inc, 4=Téc_comp, 5=Univ_inc, 6=Univ_comp, 7=Postgrado
df['educa'] = np.select([
    df['e6a_cat'] == 'Otro',
    df['e6a_cat'] == 'Basica',
    df['e6a_cat'] == 'Media',
    (df['e6a_cat'] == 'Tecnico') & (df['e6c_completo'] == 1),
    (df['e6a_cat'] == 'Tecnico') & (df['e6c_completo'] == 2),
    (df['e6a_cat'] == 'Universitaria') & (df['e6c_completo'] == 1),
    (df['e6a_cat'] == 'Universitaria') & (df['e6c_completo'] == 2),
    (df['e6a_cat'] == 'Postgrado') & (df['e6c_completo'] == 1),
    (df['e6a_cat'] == 'Postgrado') & (df['e6c_completo'] == 2)
], [np.nan, 1, 2, 4, 3, 6, 5, 7, 6], default=np.nan)

# Etiquetas
_elabels = {1: 'Básica', 2: 'Media', 3: 'Téc_inc', 4: 'Téc_comp',
             5: 'Univ_inc', 6: 'Univ_comp', 7: 'Postgrado'}
df['edu_nivel'] = df['educa'].map(_elabels)
df['edu_grupo'] = pd.cut(
    df['educa'].fillna(-1),
    bins=[-2, 2.5, 4.5, 7.5],
    labels=['Básico/Media (1-2)', 'Técnico (3-4)', 'Superior (5-7)']
)
df.loc[df['educa'].isnull(), 'edu_grupo'] = np.nan

print("educa construida:")
print(df['educa'].value_counts().sort_index().to_dict())
print()
print("Nivel educativo (etiquetas):")
print(df['edu_nivel'].value_counts().to_string())

In [ ]:
# Ingreso laboral (solo para ocupados)
# yoprcor: ingreso ocupación principal, en CLP corrientes/mes
df['ingreso_trabajo'] = df['yoprcor'].copy()

# Horas semanales trabajadas (o10)
df['horas_semana'] = df['o10'].copy()
# Verificar rango razonable (0 < horas <= 168)
df.loc[df['horas_semana'] <= 0, 'horas_semana'] = np.nan
df.loc[df['horas_semana'] > 168, 'horas_semana'] = np.nan

# Ingreso por hora (mensual / horas_mes)
# horas_mes = horas_semana * 4.33 (semanas promedio por mes)
df['ingreso_hora'] = df['ingreso_trabajo'] / (df['horas_semana'] * 4.33)

# Controlar outliers extremos (percentil 99)
p99 = df['ingreso_hora'].quantile(0.99)
df.loc[df['ingreso_hora'] > p99, 'ingreso_hora'] = np.nan

# Logaritmo del ingreso por hora (distribución más simétrica)
df['log_ingreso_hora'] = np.log(df['ingreso_hora'].replace(0, np.nan))

print("Ingreso laboral mensual (yoprcor):")
print(df['ingreso_trabajo'].describe().round(0))
print()
print("Ingreso por hora calculado:")
print(df['ingreso_hora'].describe().round(2))

In [ ]:
# Verificar brecha inicial de ingreso por hora
print("=== INGRESO POR HORA POR CONDICIÓN MIGRATORIA ===")
ing_por_mig = df.groupby('migrante')['ingreso_hora'].agg(['median','mean','count'])
ing_por_mig.index = ['Chilenos','Migrantes']
print(ing_por_mig.round(0))
print()
brecha = (ing_por_mig.loc['Migrantes','median'] / ing_por_mig.loc['Chilenos','median'] - 1) * 100
print(f"Brecha mediana (migrante vs chileno): {brecha:.1f}%")

## 6. Variables de formalidad y protección laboral

In [ ]:
# Contrato de trabajo (para dependientes)
# contrato: 1=Sí, 0=No
df['tiene_contrato'] = df['contrato'].map({1: 1, 0: 0})

# Cotización previsional (sistema de pensiones)
# cotiza: 1=Sí, 0=No
df['cotiza'] = df['cotiza'].map({1: 1, 0: 0})

# Formalidad compuesta: tiene contrato O cotiza (para mayor cobertura)
df['formal'] = np.nan
mask_ocup = df['activ'] == 1
# Formal si tiene contrato o cotiza (al menos uno)
df.loc[mask_ocup, 'formal'] = (
    (df.loc[mask_ocup, 'tiene_contrato'].fillna(0) == 1) |
    (df.loc[mask_ocup, 'cotiza'].fillna(0) == 1)
).astype(int)

print("Contrato:", df['tiene_contrato'].value_counts().to_dict())
print("Cotiza:", df['cotiza'].value_counts().to_dict())
print("Formal (contrato O cotiza, ocupados):", df['formal'].value_counts().to_dict())
print(f"Tasa formalidad: {df['formal'].mean()*100:.1f}%")

In [ ]:
# Categoría ocupacional (o15)
# 1=Empleador, 2=Cuenta_propia, 3=Público, 4=Empresa_pública, 5=Privado, 7=Doméstico
cat_map = {1:'Empleador',2:'Cuenta_propia',3:'Sector_público',
           4:'Empresa_pública',5:'Sector_privado',7:'Doméstico',8:'FF_AA',9:'Otro'}
df['cat_ocupacion'] = df['o15'].map(cat_map)

# Grupo ocupacional CIUO-08 (oficio1_08)
ocup_map = {
    1:'Directivos', 2:'Profesionales', 3:'Tecnicos', 4:'Admin',
    5:'Servicios', 6:'Agro', 7:'Artesanos', 8:'Operadores', 9:'Elementales'
}
df['ocup_grupo'] = df['oficio1_08'].map(ocup_map)

# Rama económica agrupada (rama1)
rama_map = {
    1:'Agro', 2:'Minería', 3:'Manufactura', 4:'Electricidad',
    5:'Agua', 6:'Construcción', 7:'Comercio', 8:'Transporte',
    9:'Alojamiento', 10:'Información', 11:'Finanzas', 12:'Inmobiliario',
    13:'Prof_técnico', 14:'Administrativo', 15:'AAPP', 16:'Educación',
    17:'Salud', 18:'Arte', 19:'Otros_serv', 20:'Hogares', 21:'Extraterritorial'
}
df['rama_eco'] = df['rama1'].map(rama_map)

print("Categoría ocupacional:")
print(df['cat_ocupacion'].value_counts().to_string())

## 7. Variables de pobreza y bienestar

In [ ]:
# Pobreza monetaria
# 1=Extrema, 2=No_extrema, 3=No_pobre
pobreza_map = {1: 'Extrema', 2: 'No_extrema', 3: 'No_pobre'}
df['pobreza_str'] = df['pobreza'].map(pobreza_map)
df['en_pobreza'] = (df['pobreza'].isin([1, 2])).astype(int)

# Pobreza multidimensional
# 0=No_pobre, 1=Pobre_multi
df['pobreza_multi_bn'] = df['pobreza_multi'].map({0: 0, 1: 1})

print("Pobreza monetaria:")
print(df['pobreza_str'].value_counts().to_string())
print()
print("Pobreza multidimensional:")
print(df['pobreza_multi_bn'].value_counts().to_string())

## 8. Índice de Inserción Laboral Multidimensional (IILM)

In [ ]:
# El IILM mide la calidad de la inserción laboral en 5 dimensiones
# Solo se calcula para personas en edad de trabajar (15+) y principalmente para ocupados

df_et = df[df['edad'] >= 15].copy()

# Dimensión 1: Acceso al empleo (estar ocupado)
df_et['dim_empleo'] = (df_et['activ'] == 1).astype(float)

# Dimensión 2: Formalidad (contrato o cotización)
df_et['dim_formalidad'] = df_et['formal'].astype(float)

# Dimensión 3: Ingreso (sobre la mediana del ingreso por hora de ocupados)
mediana_ing = df_et.loc[df_et['activ']==1, 'ingreso_hora'].median()
df_et['dim_ingreso'] = np.nan
df_et.loc[df_et['activ']==1, 'dim_ingreso'] = (
    df_et.loc[df_et['activ']==1, 'ingreso_hora'] >= mediana_ing
).astype(float)

# Dimensión 4: Protección social (cotiza)
df_et['dim_proteccion'] = df_et['cotiza'].astype(float)

# Dimensión 5: Adecuación (no sobrecalificado - aprox: edu superior en ocupación no elemental)
edu_sup = df_et['educa'].isin([5, 6, 7])  # Univ_inc, Univ_comp, Postgrado
ocup_elem = df_et['oficio1_08'] == 9
df_et['dim_adecuacion'] = np.nan
df_et.loc[df_et['activ']==1, 'dim_adecuacion'] = (
    ~(edu_sup & ocup_elem)
).loc[df_et['activ']==1].astype(float)

print(f"Mediana ingreso por hora usada para IILM: {mediana_ing:,.0f} CLP")
print()
print("Dimensiones del IILM (promedios):")
dims = ['dim_empleo','dim_formalidad','dim_ingreso','dim_proteccion','dim_adecuacion']
for d in dims:
    print(f"  {d}: {df_et[d].mean():.3f}")

In [ ]:
# IILM = promedio de dimensiones disponibles (para ocupados: todas 5; para no-ocupados: solo empleo=0)
df_et['iilm'] = df_et[dims].mean(axis=1)

# Baja inserción: IILM < 0.4
df_et['baja_insercion'] = (df_et['iilm'] < 0.4).astype(int)

print("Índice de Inserción Laboral Multidimensional (IILM):")
print(df_et['iilm'].describe().round(3))
print()
print("IILM por condición migratoria:")
print(df_et.groupby('migrante')['iilm'].agg(['mean','median','std']).round(3))

## 9. Selección de variables finales y guardado

In [ ]:
VARS_FINALES = [
    'id_persona', 'expr',
    # Sociodemográficas
    'sexo', 'sexo_str', 'edad', 'grupo_edad', 'region', 'macrozona',
    # Migración
    'migrante', 'lugar_nac',
    # Educación
    'educc', 'e6a_cat', 'educa', 'edu_nivel', 'edu_grupo',
    # Actividad
    'activ', 'ocupado', 'desocupado',
    # Ingresos
    'ingreso_trabajo', 'horas_semana', 'ingreso_hora', 'log_ingreso_hora',
    # Formalidad y protección
    'tiene_contrato', 'cotiza', 'formal',
    # Ocupación
    'cat_ocupacion', 'ocup_grupo', 'rama_eco',
    # Pobreza
    'pobreza', 'pobreza_str', 'en_pobreza', 'pobreza_multi', 'pobreza_multi_bn',
    # IILM
    'dim_empleo','dim_formalidad','dim_ingreso','dim_proteccion','dim_adecuacion',
    'iilm','baja_insercion',
]
# Filtrar a edad de trabajar para el dataset analítico
VARS_DISPONIBLES = [v for v in VARS_FINALES if v in df_et.columns]
df_final = df_et[VARS_DISPONIBLES].copy()

print(f"CASEN procesada (15+ años): {df_final.shape[0]:,} filas × {df_final.shape[1]} columnas")

In [ ]:
out_path = os.path.join(OUT_DATA, 'casen_procesada.parquet')
df_final.to_parquet(out_path, index=False)
print(f"CASEN procesada guardada: {out_path}")
print(f"Tamaño: {os.path.getsize(out_path)/1024:.0f} KB")

out_csv = os.path.join(OUT_DATA, 'casen_procesada.csv')
df_final.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f"CASEN procesada también en CSV: {out_csv}")

In [ ]:
# Verificación final
df_check = pd.read_parquet(out_path)
print("Verificación de lectura:", df_check.shape)
print()
print("Resumen migrantes vs chilenos (CASEN, 15+):")
resumen = df_check.groupby('migrante').agg(
    n=('activ','count'),
    pct_ocupado=('ocupado','mean'),
    pct_formal=('formal','mean'),
    ing_hora_mediana=('ingreso_hora','median'),
    iilm_medio=('iilm','mean')
).round(3)
resumen.index = ['Chilenos','Migrantes']
print(resumen.to_string())